# Matrix NTRU - Mays idea of dimension redution to attack its lattice

## Load functions

In [2]:
load("matrix_ntru_system.sage")
import numpy as np
import random
def matrixNTRULattice(H,n,p,q):
    # returns matrix NTRU corresponding lattice from public key
    AB = np.concatenate((identity_matrix(n), p.inverse_mod(q) * H), axis=1)
    CD = np.concatenate((0*identity_matrix(n), q*identity_matrix(n)), axis=1)
    L = matrix(ZZ,np.concatenate((AB,CD),axis=0))
    return L

def count_corresponding_lines(A, B, verbose=False):
    """
    This function counts the number of lines in B that correspond 
    to lines in A.

    Args:
      A: A 2D numpy array representing matrix A.
      B: A 2D numpy array representing matrix B.
      comparison_method: A function that takes two lines (rows) 
      as input and returns True if they correspond and 
      False otherwise.

    Returns:
      An integer representing the number of corresponding lines in B.
    """
    count = 0
    for lineA in A:
        for lineB in B:
            #print(lineA,lineB,lineA == lineB or lineA == -lineB)
            if lineA == lineB:
                count += 1
                if verbose:
                    print(str(lineA) + " found in target matrix")
            elif lineA == -lineB:
                if verbose:
                    print(str(lineA) + " found in target matrix (negative of it)")
            count += int(lineA == lineB or lineA == -lineB)
    return count


def matrixNTRULatticeAttack(H,n,p,q,X,Y):
    """
    creates lattice and reduces it. Afterwards, returns how 
    many lines of private key pair (X|Y) is contained in the 
    reduced lattice
atic 
    Returns:
    [a,b] where
    a: lines of private key pair (X|Y) is contained in the reduced lattice
    b: 0 if bkz terminated sucessfully in executing, and 1 if it failed
    """
    L = matrixNTRULattice(H,n,p,q)
    try:
        LBKZ = L.BKZ()
        XY = matrix(ZZ,np.concatenate((X,Y),axis=1))
        return([count_corresponding_lines(XY,LBKZ),0])
    except:
        # first line says we found zero lines and second means, bkz failed
        return([0,1])


## Toy example to see if things working 

In [3]:
# set parameters and generate public prive key pair
n = 5
p = 3
q = 64 # minimum value of n that zero decryption failure theorem works. 
setParameters([n,q]) # parameters n and q.
X,Y,Xp,Xq,H = keygen() # generate key pair


In [4]:
# use it to create lattice and lattice with dimensions in g cutted
L = matrixNTRULattice(H,n,p,q)
cut = 1
Lmay = L[:,range(0,2*n-cut)]

In [5]:
# print both lattice basis
print(L)

print(Lmay)

[ 1  0  0  0  0 37 19  9 63 10]
[ 0  1  0  0  0 27 45 54 63 55]
[ 0  0  1  0  0  9 36 18  0 18]
[ 0  0  0  1  0 47 55 28 63 28]
[ 0  0  0  0  1 45 55 27  1 26]
[ 0  0  0  0  0 64  0  0  0  0]
[ 0  0  0  0  0  0 64  0  0  0]
[ 0  0  0  0  0  0  0 64  0  0]
[ 0  0  0  0  0  0  0  0 64  0]
[ 0  0  0  0  0  0  0  0  0 64]
[ 1  0  0  0  0 37 19  9 63]
[ 0  1  0  0  0 27 45 54 63]
[ 0  0  1  0  0  9 36 18  0]
[ 0  0  0  1  0 47 55 28 63]
[ 0  0  0  0  1 45 55 27  1]
[ 0  0  0  0  0 64  0  0  0]
[ 0  0  0  0  0  0 64  0  0]
[ 0  0  0  0  0  0  0 64  0]
[ 0  0  0  0  0  0  0  0 64]
[ 0  0  0  0  0  0  0  0  0]


In [6]:
# apply BKZ to Lcut to see if we can find some lines of the public key X
LBKZ = Lmay.BKZ()[cut:,range(n)]
LBKZ

[ -1   0  -1   1   0]
[  0  -1   1   0  -1]
[ -1   1  -1   0  -1]
[  1   1   0   0   0]
[  1   0  -1  -1  -1]
[  3  13   5  -8   7]
[-10  13  10  -2  -1]
[ -5  11   3 -10   2]
[ 17   6  -5  16 -12]

In [7]:
# print matrix X (private key and manually check if it contains some lines of X or -lines)
X

[ 1  0  1 -1  0]
[ 1  0  0  1  1]
[ 1 -1  1  0  1]
[ 0  1  1  1  1]
[ 0  1 -1  0  1]

In [ ]:
# RESULTS
# line 0 of X is the negative of line 2 of LBKZ
# line 2 of X is line 0 of LBKZ
# line 3 of X is line 2 of LBKZ

In [ ]:
# QUESTION, was this by chance ? In other words, when I generate a random invertible X mod p, what is the probability of 
# of it to contain at least two lines of X ? 
total = 1000
qtd_lines = []
for _ in range(total):
    X2,Y2,Xp2,Xq2,H2 = keygen()
    qtd_lines.append(count_corresponding_lines(X,X2))

In [8]:
import collections
collections.Counter(qtd_lines)

NameError: name 'qtd_lines' is not defined

## ERASE THYIS SINCE IT ISN IN THE OTHER FILE

For a fixed n we try to attack the lattice using cut in {0,1,2,3,...,n-1} we record a vector


`(n,seed,cut,bkz_run_sucessfully,lines_found)`

In [6]:
n = 120
p = 3
q = 4096



In [70]:
p = 3
q = 4096
n = 5
cut = 0
seed_id = 0
def may_attack_fixed_seed(n,p,q,cut,seed_id):
    """
    DESCRIPTION:
        given the parameters for the matrix ntru system and a fixed seed, we generate public private key 
        and attack the private key using the corresponding lattice formed using only public paramters, 
        we reduce its dimension using the cut parameter (Mays idea) and use the returned reduced lattice basis
        and simply check how many lines of the private key X are contained in this new nxn matrix
    INPUT:
        n,p,q: matrix ntru parameters
        cut: cut used in may attack in {0,n-1}. Cut = 0 means we are not using May idea
    RETURN:
        bkz_run_sucessfully: Boolean indicating if BKZ could be run in the lattice generated by the public system
        lines_found: 0 if bkz_run_sucessfully == False and a value in {0,n} indicating how many lines of private
        key X are contained in a n-n matrix taken from the upper left corner returned by bkz. 
    NOTE:
        In case cut > 0, we need to skip the first cut lines of the matrix returned by BKZ. 
    """
    setParameters([n,q])
    set_random_seed(seed_id)
    X,Y,Xp,Xq,H = keygen()
    L = matrixNTRULattice(H,n,p,q)
    Lmay = L[:,range(0,2*n-cut)]
    try:
        LBKZ = Lmay.BKZ()
        bkz_run_sucessfully = True
        lines_found = count_corresponding_lines(X,LBKZ[range(cut,n+cut),range(n)])
    except:
        bkz_run_sucessfully = False
        lines_found = 0

    return bkz_run_sucessfully,lines_found




def run_attack_and_save_results(n = 30,p = 3,q = 4096,cut_start = 0,
                                cut_end = 2,seed_id_start = 0,seed_id_end = 1):
    """
    DESCRIPTION:
        Run attack for fixed n,p,q and a range of cuts and seeds specified above
    INPUT:
       n,p,q: matrix ntru parameters
       cut_start,cut_end: used to set range for cuts to use in the attack
       seed_id_start,seed_id_end: used to set range for cuts to use in the attack
    RETURN:
        Write on a file with a random name at folder results_may in a format
        n,p,q,cut,seed_id,bkz_run_sucessfully,lines_found
    """
    filename = "results_may/results_" + str(randrange(10^30)) + ".csv"
    with open(filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        for cut in range(cut_start,cut_end+1):
            for seed_id in range(seed_id_start,seed_id_end+1):
                bkz_run_sucessfully,lines_found = may_attack_fixed_seed(n,p,q,cut,seed_id)
                output = n,p,q,cut,seed_id,bkz_run_sucessfully,lines_found
                writer.writerow(output)

In [69]:
run_attack_and_save_results(n = 30,p = 3,q = 4096,cut_start = 0,cut_end = 10,seed_id_start = 0,seed_id_end = 3)

NameError: name 'run_attack_and_save_results' is not defined

In [44]:
def save_results(res):
    fname = "results/results_" + str(randrange(10^40)) + ".csv"
    with open(fname,"wb") as f:
        pickle.dump(res,f)

True

For n = 120
cut = 48 or cut = 46 we found 6 in 100 reps.

For n = 115
p = 3
q = 4096 and a cut of 33 we can run the BKZ algorithm, and it finds all lines of X. 

In [54]:
import csv

# Define your function that returns values (par1, par2, par3)
def my_function(par1, par2, par3):
    # This is a sample function. Replace it with your actual function logic.
    result = par1 + par2 + par3
    return par1, par2, par3, result

# Define the function to append data to CSV
def append_to_csv(par1, par2, par3):
    
    
    # Run the function to get parameters and result
    values = my_function(par1, par2, par3)

    # Open the file in append mode
    with open(filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        # Append the values as a new row in the CSV
        writer.writerow(values)

# Example usage
append_to_csv(1, 2, 3)  # Run 1
append_to_csv(4, 5, 6)  # Run 2
append_to_csv(7, 8, 9)  # Run 3


In [68]:
n=30
p = 3
q = 4096
cut_start = 0 
cut_end = 2 
seed_id_start = 0 
seed_id_end = 2




SyntaxError: expected ':' (3230355888.py, line 9)

In [66]:
list(may_attack_fixed_seed(n,p,q,cut,seed_id)

TypeError: unsupported operand parent(s) for +: '<class 'tuple'>' and 'Integer Ring'